In [1]:
import os
from dotenv import load_dotenv
load_dotenv()


groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key


'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'

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="Gemma2-9b-It", groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x0000023DA4AFD890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023DA4AFFF10>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:

# Langsmith tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")


In [5]:
from langchain_core.messages import HumanMessage
model.invoke(
    [
        HumanMessage( content="Hi im aryan and im a AI engineer")
    ]) 

AIMessage(content="Hello Aryan,\n\nIt's nice to meet you! Being an AI engineer is fascinating. What kind of projects are you working on? \n\nI'm always eager to learn more about the work that people are doing in the field of AI.\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 18, 'total_tokens': 72, 'completion_time': 0.098181818, 'prompt_time': 0.001921395, 'queue_time': 0.232120514, 'total_time': 0.100103213}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-f6f9d3d0-2b17-4d4a-9dd4-bdddf78fab10-0', usage_metadata={'input_tokens': 18, 'output_tokens': 54, 'total_tokens': 72})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage( content="Hi im aryan and im a AI engineer"),
        AIMessage( content="Hello Aryan,\n\nIt's nice to meet you! Being an AI engineer is fascinating. What kind of projects are you working on? \n\nI'm always eager to learn more about the work that people are doing in the field of AI."),
        HumanMessage(content="Hey whats my name and what do i do?")
    ])

AIMessage(content="You said your name is Aryan and that you are an AI engineer! 😊  \n\nIs there anything else you'd like to tell me about yourself or your work?  I'm curious to learn more. \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 89, 'total_tokens': 136, 'completion_time': 0.085454545, 'prompt_time': 0.005090159, 'queue_time': 0.23566290899999998, 'total_time': 0.090544704}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-64ddc439-e2ad-49e3-8855-59bdcf7f86e7-0', usage_metadata={'input_tokens': 89, 'output_tokens': 47, 'total_tokens': 136})

### message history
we can use a messae hiostory class to wrap our model and make it stateful. this will kepp iput and output messages in memory and pass them to the model as context. and store them in some datastore . Future interactions will then load those messages and pass them into the chain as a part of the  input. 

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {} ##store is a dictionary

def get_session_history(session_id: str) -> BaseChatMessageHistory: ## the return type of this function is basechatmessagehistory
    if session_id not  in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [8]:
config = {"configurable":{"session_id":"chat1"}} 

In [11]:
response=with_message_history.invoke(
    [
        HumanMessage( content="Hi im aryan and im a AI engineer")
    ], config=config) 

In [12]:
response.content

"Hi Aryan!\n\nIt's great to meet you. Being an AI engineer is such a fascinating field!  \n\nWhat are you working on these days that you're most excited about?  Do you have a particular area of AI that you specialize in?  \n\n"

In [14]:
with_message_history.invoke(
    [
        HumanMessage( content="WHat is my name ?")],
        config=config
    )


AIMessage(content='Your name is Aryan! 😊  I remember you telling me at the beginning of our conversation.  \n\n\n\nHow can I help you today, Aryan?  \n\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 255, 'total_tokens': 290, 'completion_time': 0.063636364, 'prompt_time': 0.012046027, 'queue_time': 0.23417516100000002, 'total_time': 0.075682391}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-54307712-a9ca-4468-accc-bc9c3bcac0ba-0', usage_metadata={'input_tokens': 255, 'output_tokens': 35, 'total_tokens': 290})

In [15]:
##change the config --> session id
config1 = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [
        HumanMessage( content="whats my name ")
    ], config=config1) 
response.content


"As an AI, I have no memory of past conversations and I don't know your name.\n\nIf you'd like to tell me your name, I'd be happy to know! 😊  \n\n"

In [16]:
response = with_message_history.invoke(
    [
        HumanMessage( content="my name is katoch  ")
    ], config=config1) 
response.content

"It's nice to meet you, Katoch!  \n\nIs there anything I can help you with today? 😊  \n"

In [17]:
response = with_message_history.invoke(
    [
        HumanMessage( content="whats my name ")
    ], config=config1) 
response.content

'Your name is Katoch! I remember that you told me earlier.  \n\nDo you have any other questions for me?  😊\n'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the quetions to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt | model

In [20]:
chain.invoke({"messages":[HumanMessage(content="hi"),HumanMessage(content="Hi im aryan katoch ")]})



AIMessage(content="Hi Aryan Katoch! It's nice to meet you.  \n\nHow can I help you today? 😊 \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 44, 'total_tokens': 71, 'completion_time': 0.049090909, 'prompt_time': 0.003372195, 'queue_time': 0.232086434, 'total_time': 0.052463104}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-67574c10-c9b5-4a91-81dd-efc7d9639917-0', usage_metadata={'input_tokens': 44, 'output_tokens': 27, 'total_tokens': 71})

In [21]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [22]:
config = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [
        HumanMessage( content="hi my name is aryan katoch ")
    ], config=config) 
response

AIMessage(content="Hello Aryan Katoch!  It's nice to meet you. \n\nI'm ready to answer your questions to the best of my ability.  \n\nWhat can I help you with today? 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 35, 'total_tokens': 82, 'completion_time': 0.085454545, 'prompt_time': 0.002499526, 'queue_time': 0.232054683, 'total_time': 0.087954071}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run-51c82e98-8551-465f-9b1d-e2bcb59e227c-0', usage_metadata={'input_tokens': 35, 'output_tokens': 47, 'total_tokens': 82})

In [23]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [24]:
response = chain.invoke({"messages": [HumanMessage(content="Hi im aryan katoch")], "language": "Hindi"})
response.content


'नमस्ते आर्यन् कटोच! 👋  मुझे आपसे बात करने में खुशी हो रही है। 😊 \n\nआप मुझसे कोई भी सवाल पूछ सकते हैं, मैं अपनी पूरी कोशिश करूँगा कि आपको मदद कर सकूं। 👍 \n\n'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [25]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
    )

In [26]:
config = {"configurable":{"session_id":"chat4"}}
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="hi im aryan katoch I'm an AI engineer ")], 'language':'Hindi'},
     config=config) 
response.content

'नमस्ते अर्यान कटोच! यह जानकर बहुत अच्छा लगा कि आप एक एआई इंजीनियर हैं। \n\nमुझे आपकी मदद करने में खुशी होगी। आपके कोई सवाल हैं?  \n\n'

In [27]:
response = with_message_history.invoke(
    {'messages':[HumanMessage(content="What's my name?")], 'language':'Hindi'},
     config=config) 
response.content

'आपका नाम अर्यान कटोच है।  \n'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.


'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [29]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy = "last",
    token_counter=model,
    include_system=True,
    allow_partial=True,
    start_on ="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={})]

In [31]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain =(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model
)


response=chain.invoke(
    {
    "messages":messages +[ HumanMessage(content="what math problem i asked?")],
    "language":"hindi"
    }
)
response.content

'आपने मुझसे 2 + 2 का जवाब पूछा था।  \n'

In [32]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}